# Chatbot RAG — Sổ tay sinh viên HVCS

Chatbot nhẹ, chạy hoàn toàn trên CPU. Hệ thống đọc các PDF trong thư mục `data/`, tìm các đoạn liên quan bằng TF-IDF và trả lời **chỉ dựa trên sổ tay**, kèm số trang để kiểm chứng.

> Lưu ý: Đây là RAG trích xuất (extractive RAG), không cần API/LLM trả phí. Câu trả lời là các đoạn nguyên văn phù hợp nhất thay vì suy diễn ngoài tài liệu.

## 1. Cài thư viện (chạy một lần)

Nếu chạy trên Google Colab/Jupyter mới, bỏ comment và chạy cell dưới đây.

In [1]:
# Nếu đang chạy Jupyter tích hợp trong Spyder, dùng lệnh này để tránh xung đột jedi của Spyder:
#%pip install -q gradio pdfplumber scikit-learn "jedi>=0.17.2,<0.20.0" "ipykernel>=6.29.3,<7"
# Sau khi cài xong, khởi động lại kernel nếu Jupyter yêu cầu.

## 2. Đọc và chia nhỏ tài liệu

Đặt PDF sổ tay trong `data/`. Notebook tự quét mọi PDF trong thư mục này; vì vậy có thể thay/tăng tài liệu mà không cần sửa code.

In [2]:
from pathlib import Path
import re
import pdfplumber

DATA_DIR = Path('data')
CHUNK_SIZE = 1_100       # ký tự/đoạn — đủ ngữ cảnh, vẫn nhẹ
CHUNK_OVERLAP = 180

def normalise(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

def split_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Cắt có overlap; ưu tiên kết thúc ở dấu câu để dễ đọc."""
    text = normalise(text)
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        if end < len(text):
            boundary = max(text.rfind('. ', start, end), text.rfind('; ', start, end), text.rfind(': ', start, end))
            if boundary > start + chunk_size // 2:
                end = boundary + 1
        chunk = text[start:end].strip()
        if len(chunk) >= 80:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

def load_documents(data_dir=DATA_DIR):
    pdfs = sorted(data_dir.glob('*.pdf'))
    if not pdfs:
        raise FileNotFoundError(f'Không tìm thấy PDF nào trong: {data_dir.resolve()}')

    records, skipped_pages = [], []
    for pdf_path in pdfs:
        with pdfplumber.open(pdf_path) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text() or ''
                if not normalise(page_text):
                    skipped_pages.append((pdf_path.name, page_number))
                    continue
                for chunk in split_text(page_text):
                    records.append({
                        'text': chunk,
                        'source': pdf_path.name,
                        'page': page_number,
                    })
    if not records:
        raise ValueError('PDF không có văn bản có thể trích xuất. Hãy OCR tài liệu trước.')
    return records, skipped_pages

documents, skipped_pages = load_documents()
print(f'Đã lập kho tri thức: {len(documents)} đoạn từ {len(set(d["source"] for d in documents))} PDF.')
if skipped_pages:
    print(f'Bỏ qua {len(skipped_pages)} trang không có text (có thể là trang ảnh).')

Đã lập kho tri thức: 526 đoạn từ 1 PDF.


## 3. Tạo chỉ mục và hàm RAG

TF-IDF là bộ embedding nhẹ, không phải tải mô hình lớn. Ngưỡng tin cậy giúp chatbot nói rõ khi sổ tay không có thông tin liên quan.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2), sublinear_tf=True, max_df=0.98
)
matrix = vectorizer.fit_transform([item['text'] for item in documents])

def retrieve(question: str, top_k: int = 3):
    query = normalise(question)
    if not query:
        return []
    scores = cosine_similarity(vectorizer.transform([query]), matrix).ravel()
    best_ids = scores.argsort()[::-1][:top_k]
    return [{**documents[i], 'score': float(scores[i])} for i in best_ids if scores[i] > 0]

MIN_RELEVANCE = 0.055

def answer_from_handbook(question: str) -> str:
    results = retrieve(question)
    if not results or results[0]['score'] < MIN_RELEVANCE:
        return ('Mình chưa tìm thấy nội dung đủ liên quan trong Sổ tay sinh viên. '
                'Bạn hãy thử nêu rõ hơn từ khóa (ví dụ: học phí, đăng ký học phần, ký túc xá).')

    answer = ['**Thông tin tìm thấy trong sổ tay:**']
    for i, item in enumerate(results, 1):
        excerpt = item['text']
        answer.append(
            f'\n**{i}. {item["source"]} — trang {item["page"]}**\n> {excerpt}'
        )
    answer.append('\n*Vui lòng đối chiếu số trang nguồn khi cần áp dụng quy định cụ thể.*')
    return '\n'.join(answer)

# Kiểm tra nhanh retrieval:
print(answer_from_handbook('quy định đăng ký học phần')[:1200])

**Thông tin tìm thấy trong sổ tay:**

**1. So tay Sinh vien (2026) HVCS.pdf — trang 75**
> g thông báo điều chỉnh của Học viện. Kết quả đăng ký học của mỗi sinh viên được thông báo trên Phiếu đăng ký học tập và trên tài khoản cá nhân trực tuyến của sinh viên đã được cấp. Trong đó, có các thông tin về học phần, số tín chỉ của mỗi học phần và lịch học. 6. Điều kiện đăng ký học tập: Điều kiện đăng ký từng học phần được quy định cụ thể trong chương trình đào tạo. Ngoài ra, sinh viên phải hoàn thành nghĩa vụ học phí của học kỳ hiện tại mới được đăng ký học tập cho học kỳ tới. 7. Sinh viên phải đăng ký học lại các học phần bắt buộc có điểm tổng kết không đạt yêu cầu ở các học kỳ trước. Đối với các học phần tự chọn không đạt, sinh viên được phép đăng 69

**2. So tay Sinh vien (2026) HVCS.pdf — trang 76**
> ký học lại học phần đó hoặc chọn học phần khác thay thế trong số các học phần tự chọn quy định trong mỗi chương trình đào tạo. 8. Sinh viên được phép đăng ký học lại các học phần đã đạt để 

## 4. Web chatbot responsive (desktop và mobile)

Chạy cell này để mở giao diện. Trên máy cá nhân, truy cập URL hiện ra; trên điện thoại cùng Wi-Fi, dùng địa chỉ mạng LAN.

In [6]:
import os
import gradio as gr

CSS = """
#app {max-width: 940px; margin: auto;}
.gradio-container {font-family: system-ui, -apple-system, 'Segoe UI', sans-serif;}
#title {text-align:center; margin-bottom: 0;}
@media (max-width: 600px) {
  .gradio-container {padding: 8px !important;}
  #title {font-size: 1.35rem;}
  .message {font-size: .95rem !important;}
}
"""

def chat(message, history):
    message = (message or '').strip()
    if not message:
        return '', history
    reply = answer_from_handbook(message)
    history = history or []
    history = history + [
        {'role': 'user', 'content': message},
        {'role': 'assistant', 'content': reply},
    ]
    return '', history

with gr.Blocks(title='Chatbot Sổ tay sinh viên') as app:
    gr.Markdown('# Chatbot Sổ tay sinh viên', elem_id='title')
    gr.Markdown('Hỏi đáp dựa trên tài liệu trong kho tri thức. Câu trả lời luôn kèm trang nguồn.')
    # Gradio 6 mặc định dùng định dạng messages; không truyền type để tương thích 6.0.
    chatbot = gr.Chatbot(height=480, elem_id='app')
    with gr.Row():
        question = gr.Textbox(
            placeholder='Ví dụ: Em cần làm gì để đăng ký học phần?',
            label='Câu hỏi', scale=8
        )
        send = gr.Button('Gửi', variant='primary', scale=1)
    gr.Examples(
        ['Quy định đăng ký học phần là gì?', 'Học phí được quy định như thế nào?', 'Thông tin về ký túc xá'],
        inputs=question, label='Câu hỏi gợi ý'
    )
    clear = gr.ClearButton([question, chatbot], value='Xóa cuộc trò chuyện')
    send.click(chat, [question, chatbot], [question, chatbot])
    question.submit(chat, [question, chatbot], [question, chatbot])

# Render cấp PORT qua biến môi trường. Khi chạy local, để Gradio tự tìm cổng trống.
render_port = os.getenv('PORT')
launch_options = {'server_name': '0.0.0.0', 'share': False, 'css': CSS}
if render_port:
    launch_options['server_port'] = int(render_port)
app.launch(**launch_options)

ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7860): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

## 5. Deploy miễn phí trên Render

1. Đẩy toàn bộ thư mục (gồm notebook và `data/`) lên một GitHub repository.
2. Trên Render tạo **Web Service**, chọn repository đó và Python runtime.
3. Chạy cell **Xuất chỉ mục** ở cuối notebook, sau đó commit thư mục `artifacts/` cùng `app.py` và `requirements.txt`.
4. Đặt **Build Command**: `pip install -r requirements.txt`
5. Đặt **Start Command**: `python app.py`. Render tự truyền biến `PORT`; app sẽ lắng nghe đúng cổng.

> Khi cập nhật PDF, redeploy service để tạo chỉ mục mới.

In [ ]:
# Xuất chỉ mục để app.py dùng khi deploy (chạy lại cell này mỗi khi đổi PDF).
from pathlib import Path
import joblib

artifact_path = Path('artifacts') / 'rag_index.joblib'
artifact_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {'vectorizer': vectorizer, 'matrix': matrix, 'documents': documents},
    artifact_path,
    compress=3,
)
print(f'Đã xuất chỉ mục: {artifact_path} ({artifact_path.stat().st_size / 1024:.0f} KB)')